# 04 · Optimizers, schedulers, hyperparameters, curves and gradient flow
Point 4 of `PLAN.md` (owner: Mykola). Builds on point 3's winning architecture and point 5's tokenizer (BPE 16k, label on last piece) to cover everything the assignment lists under "Experiment with different Optimizers, Schedulers, hyperparameters, and all other elements... Collect all relevant results: loss/metric curves, gradient flow visualizations, parameter tuning logs":
1. **Optimizers** — AdamW (the team's baseline), Adam, RMSprop, SGD+momentum at the same lr, and SGD at a lr actually tuned for SGD.
2. **Schedulers** — none (baseline), StepLR, CosineAnnealingLR, OneCycleLR, ReduceLROnPlateau, linear warmup+decay.
3. **Hyperparameters** — one-factor-at-a-time sweeps around the best optimizer+scheduler: learning rate, weight decay, batch size, dropout, hidden size.
4. **Curves and gradient flow** — collected across all of the above, with a dedicated section on how schedulers and extreme hyperparameters change the gradient-norm trajectory, not only the final score.

**Base architecture:** loaded automatically from `results_03_architecture.csv` (point 3's best run by CV F1). If that file is not there yet (point 3 hasn't been run in this environment), this notebook falls back to point 5's recommended architecture (1-layer BiGRU) so it can still run standalone — but the real numbers should come from a run where point 3 has already finished.

**Section 0 is the same master copy as notebooks 02/03/05**, except section 0.8 below, which is new: the shared `TaggerModule` only ever builds AdamW (section 0.4), so this notebook adds a small, self-contained generalization (`OptimTaggerModule` + `run_experiment_v2`) rather than touching the master copy that the rest of the team relies on.

**How to run:** on Colab with a T4. ~5 optimizer runs + ~6 scheduler runs (1 reused) + ~13 hyperparameter runs ≈ 23 runs at roughly the point-5 BiGRU pace (4-5 min/epoch, up to 6 epochs, early stopping). **Expect on the order of 6-9 hours for the full notebook** — deliberately long, matching the unstable-power note; every run resumes from its `report.json` if interrupted, so stopping and restarting the kernel loses at most the run that was in progress. Trim any of the `*_GRID` lists in sections 2-4 to shorten it. Set `QUICK_RUN=1` to check the code on a tiny subset in a few minutes, on CPU.

In [ ]:
NOTEBOOK = "04-optimization"  # W&B group of this notebook's runs

## 0. Shared code
Section 0 is **identical in every model notebook** (copied verbatim from the master copy in `02_validation_baseline.ipynb`), so this notebook uses the same splits, metric, tokenizer, training loop and output format as points 2 and 5. Code specific to this notebook goes in its own sections below.

### 0.0 Setup
- **Colab:** upload `secrets.env` (`KAGGLE_USERNAME`, `KAGGLE_KEY`, `WANDB_API_KEY`) to `/content/` first. The cell loads the keys (it prints only their names), installs missing packages, and downloads the data with the Kaggle CLI. If Google Drive is mounted at `/content/drive`, run outputs go to `MyDrive/dl-a3-artifacts` so they survive the VM reset — make that folder a shortcut to the team's shared folder, so everyone writes to the same results log.
- **Locally:** the data is read from `data/raw/`, outputs go to `artifacts/` (git-ignored).
- `QUICK_RUN=1` (environment variable) trains on a small subset for one epoch, to check the code in a few minutes, on CPU.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

COMPETITION = "kse-ua-location-extraction-2026"
DATA_FILES = ["uk_geo_dataset.csv", "labeling_sample.csv", "test.csv"]
IN_COLAB = Path("/content").exists()
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"


def load_secrets(path: Path = Path("/content/secrets.env")) -> None:
    assert path.exists(), f"Upload secrets.env to {path.parent} first"
    for line in path.read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()
    print("keys loaded:", [k for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "WANDB_API_KEY") if os.environ.get(k)])


def download_data(raw: Path) -> None:
    raw.mkdir(parents=True, exist_ok=True)
    for name in DATA_FILES:
        if (raw / name).exists():
            continue
        subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION, "-f", name, "-p", str(raw)], check=True)
        archive = raw / f"{name}.zip"
        if archive.exists():  # large files arrive zipped
            shutil.unpack_archive(archive, raw)
            archive.unlink()


def install_missing(packages: dict[str, str]) -> None:
    """{import name: pip name}; installs only what is missing."""
    missing = [pip for module, pip in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


install_missing({"lightning": "lightning", "tokenizers": "tokenizers", "wandb": "wandb", "kaggle": "kaggle"})
if IN_COLAB:
    load_secrets()
    RAW = Path("/content/data/raw")
    download_data(RAW)
    drive = Path("/content/drive/MyDrive")
    ARTIFACTS = (drive / "dl-a3-artifacts") if drive.exists() else Path("/content/artifacts")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RAW, ARTIFACTS = ROOT / "data/raw", ROOT / "artifacts"
if QUICK_RUN:
    ARTIFACTS = ARTIFACTS / "quick_run"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"data: {RAW} | outputs: {ARTIFACTS} | quick run: {QUICK_RUN}")

### 0.1 Tokenizer, span ↔ BIO, competition metric
Every training notebook uses the same code. Each token keeps its character offsets, so predicted tags can be turned back into **exact substrings** of the text, which is what the metric compares. Punctuation becomes its own token (`Київ,` → `Київ` `,`), while inner hyphens and apostrophes stay inside the word (`Івано-Франківськ`, `Слов'янськ`).

In [ ]:
import ast
import re
import string
from collections import Counter
from dataclasses import dataclass
from typing import List, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

TOKEN_RE = re.compile(r"\w+(?:[-'’ʼ`]\w+)*|[^\w\s]")


@dataclass(frozen=True)
class Token:
    text: str
    start: int
    end: int  # exclusive


def word_tokenize(text: str) -> list[Token]:
    return [Token(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


def parse_markers(value) -> list[tuple[int, int]]:
    """`loc_markers` as stored in the CSV ("[(0, 4), (10, 15)]") -> [(start, end), ...], end exclusive."""
    if not isinstance(value, str):
        return []
    return [(int(a), int(b)) for a, b in ast.literal_eval(value.strip() or "[]")]


def spans_to_bio(tokens: Sequence[Token], spans) -> list[str]:
    """Every token that overlaps a span is labelled: the first one B-LOC, the rest I-LOC."""
    labels = ["O"] * len(tokens)
    for left, right in sorted(spans):
        first = True
        for i, tok in enumerate(tokens):
            if tok.start < right and tok.end > left:
                labels[i] = "B-LOC" if first else "I-LOC"
                first = False
    return labels


def bio_to_spans(tokens: Sequence[Token], tags: Sequence[str]) -> list[tuple[int, int]]:
    """Merge BIO tags into character spans; an I-LOC without an open entity starts a new one."""
    spans, current = [], None
    for tok, tag in zip(tokens, tags):
        if tag == "B-LOC" or (tag == "I-LOC" and current is None):
            if current is not None:
                spans.append(tuple(current))
            current = [tok.start, tok.end]
        elif tag == "I-LOC":
            current[1] = tok.end
        else:
            if current is not None:
                spans.append(tuple(current))
            current = None
    if current is not None:
        spans.append(tuple(current))
    return spans


def spans_to_strings(text: str, spans) -> list[str]:
    """Cut location strings out of the original text (deduplicated, order kept)."""
    return list(dict.fromkeys(s for s in (text[a:b].strip() for a, b in spans) if s))

`hard_processing` and `comp_metric` below are copied **verbatim** from the competition page. Do not "fix" them.

In [ ]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p,  = 0.0, 0.0, 0.0, 0.0

    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])

        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)

    if tp + fp == 0:
        if p == 0:
            precision = 1.0
        else:
            precision = 0.0
    else:
        precision = tp/(tp + fp)

    if tp + fn == 0:
        if p == 0:
            recall = 1.0
        else:
            recall = 0.0
    else:
        recall = tp/(tp + fn)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

Complementary metrics (notebook 01, section 8). They count exactly like `comp_metric`, plus:

In [ ]:
def precision_recall(y_true, y_pred) -> tuple[float, float]:
    tp = fp = fn = 0
    for t_s, p_s in zip(y_true, y_pred):
        t, p = {hard_processing(s) for s in t_s}, {hard_processing(s) for s in p_s}
        tp, fp, fn = tp + len(t & p), fp + len(p - t), fn + len(t - p)
    precision = tp / (tp + fp) if tp + fp else float(tp + fn == 0)
    recall = tp / (tp + fn) if tp + fn else 1.0
    return precision, recall


def partial_match_f1(y_true, y_pred) -> float:
    """Boundary-tolerant F1: a location counts if it shares a normalized word with one on the other side."""
    words = lambda s: set(hard_processing(s).split())
    hit_p = n_p = hit_t = n_t = 0
    for t_s, p_s in zip(y_true, y_pred):
        t_sets = [words(s) for s in {hard_processing(x) for x in t_s}]
        p_sets = [words(s) for s in {hard_processing(x) for x in p_s}]
        n_p, n_t = n_p + len(p_sets), n_t + len(t_sets)
        hit_p += sum(any(p & t for t in t_sets) for p in p_sets)
        hit_t += sum(any(t & p for p in p_sets) for t in t_sets)
    precision, recall = (hit_p / n_p if n_p else 1.0), (hit_t / n_t if n_t else 1.0)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def metric_report(y_true, y_pred) -> dict:
    precision, recall = precision_recall(y_true, y_pred)
    no_loc = [len(p) > 0 for t, p in zip(y_true, y_pred) if not t]
    with_loc = [(t, p) for t, p in zip(y_true, y_pred) if t]
    return {
        "comp_f1": comp_metric(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "partial_f1": partial_match_f1(y_true, y_pred),
        "f1_texts_with_loc": comp_metric([t for t, _ in with_loc], [p for _, p in with_loc]),
        "fp_rate_texts_without_loc": float(np.mean(no_loc)) if no_loc else 0.0,
    }


def bootstrap_ci(y_true, y_pred, n_boot: int = 2000, seed: int = 0, alpha: float = 0.05) -> tuple[float, float]:
    """95% interval of comp_metric when the texts are resampled with replacement (CV has only 26 texts)."""
    rng, n = np.random.default_rng(seed), len(y_true)
    scores = [comp_metric([y_true[i] for i in idx], [y_pred[i] for i in idx]) for idx in rng.integers(0, n, (n_boot, n))]
    low, high = np.quantile(scores, [alpha / 2, 1 - alpha / 2])
    return float(low), float(high)

### 0.2 Data: corpora and encoded datasets
- A `Corpus` holds the texts, word tokens and BIO tags (label ids) of one split, and the gold strings. For evaluation splits it also keeps the tokens with their offsets, which decoding needs (train skips them to save memory).
- A **vocabulary** maps words to ids. Any vocabulary works if it has `encode_batch(word_lists) -> [(ids, first_pos, last_pos), ...]`, where `first_pos[w]` and `last_pos[w]` are the positions of word `w`'s first and last id. For word-level they are equal. For BPE a word can have several pieces.
- `EncodedDataset` puts each word's label on its first or last piece (`alignment`). Every other position gets −100, which the loss ignores. Training sequences are cut at `max_len` ids. Evaluation sequences are **never cut**, because test posts can be long (notebook 01).

In [ ]:
import json
import logging
import time
import warnings
from collections import Counter

import lightning as L
import torch
import torch.nn as nn
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from lightning.pytorch.utilities import grad_norm
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", module="lightning")
logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["O", "B-LOC", "I-LOC"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))
PAD_TOKEN, UNK_TOKEN, PAD_ID, UNK_ID, IGNORE_INDEX = "<pad>", "<unk>", 0, 1, -100
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


class Corpus:
    def __init__(self, texts, spans=None, keep_tokens: bool = True):
        texts = [str(t) for t in texts]
        tokens = [word_tokenize(t) for t in texts]
        self.words = [[t.text for t in toks] for toks in tokens]
        self.tags = None if spans is None else [
            np.array([LABEL2ID[tag] for tag in spans_to_bio(toks, s)], dtype=np.int64) for toks, s in zip(tokens, spans)]
        self.gold = None if spans is None else [[t[a:b] for a, b in s] for t, s in zip(texts, spans)]
        self.texts, self.tokens = texts, (tokens if keep_tokens else None)

    def __len__(self) -> int:
        return len(self.words)


class WordVocab:
    """Word -> id, built on training words only; the `max_size` most frequent words seen `min_freq`+ times; the rest
    -> <unk>. `lowercase=False` keeps case (`Київ` and `київ` become different words)."""

    def __init__(self, word_lists, max_size: int = 50_000, min_freq: int = 2, lowercase: bool = True):
        self.norm = str.lower if lowercase else str
        counts = Counter(self.norm(w) for words in word_lists for w in words)
        self.itos = [PAD_TOKEN, UNK_TOKEN] + [w for w, c in counts.most_common(max_size - 2) if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self) -> int:
        return len(self.itos)

    def encode_batch(self, word_lists):
        for words in word_lists:
            ids = [self.stoi.get(self.norm(w), UNK_ID) for w in words]
            positions = list(range(len(words)))
            yield ids, positions, positions


class EncodedDataset(Dataset):
    def __init__(self, corpus: Corpus, vocab, alignment: str = "first", max_len: int | None = None):
        assert alignment in ("first", "last")
        self.items = []
        for i, (ids, first, last) in enumerate(vocab.encode_batch(corpus.words)):
            ids, pos = np.asarray(ids, dtype=np.int64), np.asarray(first if alignment == "first" else last, dtype=np.int64)
            labels = np.full(len(ids), IGNORE_INDEX, dtype=np.int64)
            if corpus.tags is not None:
                labels[pos] = corpus.tags[i]
            if max_len is not None and len(ids) > max_len:
                ids, labels, pos = ids[:max_len], labels[:max_len], pos[pos < max_len]
            self.items.append((ids, labels, pos))

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        return (idx, *self.items[idx])


def collate(batch) -> dict:
    """Right-pad ids with PAD_ID and labels with −100; `lengths` feed pack_padded_sequence."""
    lengths = torch.tensor([max(len(ids), 1) for _, ids, _, _ in batch])
    input_ids = torch.full((len(batch), int(lengths.max())), PAD_ID, dtype=torch.long)
    labels = torch.full_like(input_ids, IGNORE_INDEX)
    for row, (_, ids, lab, _) in enumerate(batch):
        input_ids[row, :len(ids)] = torch.from_numpy(ids)
        labels[row, :len(lab)] = torch.from_numpy(lab)
    return {"idx": [b[0] for b in batch], "input_ids": input_ids, "labels": labels, "lengths": lengths,
            "positions": [b[3] for b in batch]}

**Splits** (point 2.1, notebook 02). They are fixed by the data, so they need no seed:
- **train** = `uk_geo_dataset` with `is_valid==0` (1M sentences);
- **es** = `is_valid==1` (10k sentences, train domain): early stopping and checkpoint choice;
- **CV** = `labeling_sample` (26 test-like texts): the score we report and compare with the LB;
- **test** = 477 posts for the submission.

`QUICK_RUN` samples 20k train and 2k es sentences with a fixed seed.

In [ ]:
def load_corpora() -> tuple[dict[str, Corpus], pd.DataFrame]:
    uk = pd.read_csv(RAW / "uk_geo_dataset.csv", usecols=["text", "loc_markers", "is_valid"])
    ls = pd.read_csv(RAW / "labeling_sample.csv")
    test = pd.read_csv(RAW / "test.csv")
    train_df, es_df = uk[uk.is_valid == 0], uk[uk.is_valid == 1]
    if QUICK_RUN:
        train_df, es_df = train_df.sample(20_000, random_state=0), es_df.sample(2_000, random_state=0)
    corpora = {
        "train": Corpus(train_df.text, train_df.loc_markers.map(parse_markers), keep_tokens=False),
        "es": Corpus(es_df.text, es_df.loc_markers.map(parse_markers)),
        "cv": Corpus(ls.text, ls.loc_markers.map(parse_markers)),
        "test": Corpus(test.text),
    }
    return corpora, test

### 0.3 Model
Embedding → GRU / LSTM / RNN (optionally bidirectional, several layers) → a linear layer on every position. `pack_padded_sequence` makes the RNN stop at each text's real length, so padding never enters the hidden state, not even in the backward direction. This class already supports everything point 3 needs (`rnn_type`, `num_layers`, `bidirectional`) — nothing to change here.

In [ ]:
RNN_TYPES = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}


class RNNTagger(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 256, rnn_type: str = "gru",
                 num_layers: int = 1, bidirectional: bool = True, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.dropout = nn.Dropout(dropout)
        self.rnn = RNN_TYPES[rnn_type](embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                                       bidirectional=bidirectional, dropout=dropout if num_layers > 1 else 0.0)
        self.classifier = nn.Linear(hidden_dim * (2 if bidirectional else 1), len(LABELS))

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        x = self.dropout(self.embedding(input_ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=input_ids.size(1))
        return self.classifier(self.dropout(out))  # [batch, time, labels]

### 0.4 Training and decoding
- Loss: token-level cross-entropy, ignoring −100.
- After every epoch, both validation sets are decoded to strings and scored with the **competition metric**:
  - `es_f1` (train domain) selects the checkpoint and drives early stopping;
  - `cv_f1` (test-like) is only reported. Choosing epochs on 26 CV texts would overfit them.
- Gradient norms per parameter (before clipping) are logged every `log_every_n_steps` steps: the gradient-flow check this notebook relies on in section 6.
- Decoding: word-level probabilities → argmax tag per word → BIO → character spans → strings cut from the text.

In [ ]:
def word_probs(logits: torch.Tensor, positions) -> list[np.ndarray]:
    """[batch, time, labels] logits -> per text a [n_words, labels] probability array, read at the aligned piece."""
    probs = logits.float().softmax(-1).detach().cpu().numpy()
    return [probs[row, pos] for row, pos in enumerate(positions)]


def decode(corpus: Corpus, probs: list[np.ndarray]) -> list[list[str]]:
    tags = [[ID2LABEL[int(i)] for i in p.argmax(-1)] for p in probs]
    return [spans_to_strings(text, bio_to_spans(toks, t)) for text, toks, t in zip(corpus.texts, corpus.tokens, tags)]


class TaggerModule(L.LightningModule):
    def __init__(self, model: nn.Module, lr: float, weight_decay: float, val_corpora: dict[str, Corpus]):
        super().__init__()
        self.model, self.lr, self.weight_decay = model, lr, weight_decay
        self.val_names, self.val_corpora = list(val_corpora), list(val_corpora.values())
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    def _loss(self, logits, labels):
        return self.loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

    def training_step(self, batch, batch_idx):
        loss = self._loss(self.model(batch["input_ids"], batch["lengths"]), batch["labels"])
        self.log("train_loss", loss, on_step=True, on_epoch=True, batch_size=len(batch["idx"]))
        return loss

    def on_validation_epoch_start(self):
        self.val_probs = [{} for _ in self.val_names]

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        logits = self.model(batch["input_ids"], batch["lengths"])
        name = self.val_names[dataloader_idx]
        self.log(f"{name}_loss", self._loss(logits, batch["labels"]), batch_size=len(batch["idx"]),
                 add_dataloader_idx=False)
        self.val_probs[dataloader_idx].update(zip(batch["idx"], word_probs(logits, batch["positions"])))

    def on_validation_epoch_end(self):
        for name, corpus, buf in zip(self.val_names, self.val_corpora, self.val_probs):
            pred = decode(corpus, [buf[i] for i in range(len(corpus))])
            precision, recall = precision_recall(corpus.gold, pred)
            self.log_dict({f"{name}_f1": comp_metric(corpus.gold, pred), f"{name}_precision": precision,
                           f"{name}_recall": recall})

    def on_before_optimizer_step(self, optimizer):  # before clipping; Lightning writes it every log_every_n_steps
        self.log_dict(grad_norm(self.model, norm_type=2))

    def configure_optimizers(self):
        return torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)


@torch.no_grad()
def predict(model: nn.Module, dataset: EncodedDataset, batch_size: int = 256) -> list[np.ndarray]:
    model.eval().to(DEVICE)
    out = [None] * len(dataset)
    for batch in DataLoader(dataset, batch_size=batch_size, collate_fn=collate):
        logits = model(batch["input_ids"].to(DEVICE), batch["lengths"])
        for i, p in zip(batch["idx"], word_probs(logits, batch["positions"])):
            out[i] = p
    return out

### 0.5 One experiment
`run_experiment(cfg, vocab, corpora, test_ids)`: train → reload the best checkpoint (by `es_f1`) → predict CV and test → CV report → save outputs. If the run's `report.json` already exists it is loaded instead, so a crashed or disconnected session (relevant with unstable power) can resume. Outputs in `ARTIFACTS/{run_name}/`: `best.ckpt`, `config.json`, `logs/metrics.csv` (curves), `cv_probs.pkl` / `test_probs.pkl`, `submission.csv`, `report.json`.

In [ ]:
import pickle

WANDB_PROJECT = "kse-ua-location-extraction"


def save_probs(probs, path: Path) -> None:
    with open(path, "wb") as f:
        pickle.dump([p.astype(np.float16) for p in probs], f)


def load_probs(run: str, split: str) -> list[np.ndarray]:
    """Word-level probabilities a run saved for `split` ("cv" or "test")."""
    with open(ARTIFACTS / run / f"{split}_probs.pkl", "rb") as f:
        return [p.astype(np.float32) for p in pickle.load(f)]


def write_submission(text_ids, locations, path: Path) -> None:
    pd.DataFrame({"text_id": text_ids,
                  "locations": [json.dumps(list(x), ensure_ascii=False) for x in locations]}).to_csv(path, index=False)


def run_experiment(cfg: dict, vocab, corpora: dict[str, Corpus], test_ids, force: bool = False) -> dict:
    out = ARTIFACTS / cfg["run_name"]
    if (out / "report.json").exists() and not force:
        return json.loads((out / "report.json").read_text())
    out.mkdir(parents=True, exist_ok=True)
    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    L.seed_everything(cfg["seed"], workers=True)
    start = time.time()

    encode = lambda name, max_len=None: EncodedDataset(corpora[name], vocab, cfg["alignment"], max_len)
    train_ds, es_ds, cv_ds, test_ds = encode("train", cfg["max_len"]), encode("es"), encode("cv"), encode("test")
    workers = 2 if DEVICE == "cuda" else 0
    loader = lambda ds, shuffle=False: DataLoader(ds, batch_size=cfg["batch_size"], shuffle=shuffle,
                                                  collate_fn=collate, num_workers=workers)

    model = RNNTagger(len(vocab), **cfg["model"])
    module = TaggerModule(model, cfg["lr"], cfg["weight_decay"], {"es": corpora["es"], "cv": corpora["cv"]})
    loggers = [CSVLogger(ARTIFACTS, name=cfg["run_name"], version="logs")]
    if os.environ.get("WANDB_API_KEY") and not QUICK_RUN:
        loggers.append(WandbLogger(project=WANDB_PROJECT, name=cfg["run_name"], group=NOTEBOOK, config=cfg,
                                   save_dir=str(out)))
    checkpoint = ModelCheckpoint(dirpath=out, filename="best", monitor="es_f1", mode="max", save_top_k=1)
    trainer = L.Trainer(max_epochs=cfg["max_epochs"], accelerator="gpu" if DEVICE == "cuda" else "cpu", devices=1,
                        gradient_clip_val=1.0, log_every_n_steps=10 if QUICK_RUN else 100, num_sanity_val_steps=0,
                        logger=loggers,
                        callbacks=[checkpoint, EarlyStopping(monitor="es_f1", mode="max", patience=cfg["patience"])],
                        enable_progress_bar=False, enable_model_summary=False)
    trainer.fit(module, loader(train_ds, shuffle=True), [loader(es_ds), loader(cv_ds)])

    best = torch.load(checkpoint.best_model_path, map_location="cpu", weights_only=False)
    model.load_state_dict({k.removeprefix("model."): v for k, v in best["state_dict"].items()})
    cv_probs, test_probs = predict(model, cv_ds), predict(model, test_ds)
    save_probs(cv_probs, out / "cv_probs.pkl")
    save_probs(test_probs, out / "test_probs.pkl")
    write_submission(test_ids, decode(corpora["test"], test_probs), out / "submission.csv")

    report = {"run_name": cfg["run_name"], **metric_report(corpora["cv"].gold, decode(corpora["cv"], cv_probs)),
              "es_f1": float(checkpoint.best_model_score), "best_epoch": int(best["epoch"]),
              "epochs_run": trainer.current_epoch, "vocab_size": len(vocab),
              "params": sum(p.numel() for p in model.parameters()), "minutes": (time.time() - start) / 60}
    (out / "report.json").write_text(json.dumps(report, indent=2))
    if len(loggers) > 1:
        loggers[1].experiment.summary.update({f"final_cv_{k}": v for k, v in report.items() if k != "run_name"})
        loggers[1].experiment.finish()
    return report

### 0.6 Results log and submissions
`ARTIFACTS/experiments.csv` holds one row per run: `run_name, owner, point, description, cv_f1, cv_p, cv_r, es_f1, lb_public, notes`. Failed runs stay in. `log_result` replaces the run's row and keeps an LB score that is already there. `submit` sends a run's `submission.csv` to Kaggle, waits for the public score, and writes it into the log. The team has 10 submissions a day (Ivan 3, Mykola 4, Taras 3) — this notebook never submits automatically; `SUBMIT_RUN` has to be set by hand after reviewing the results.

In [ ]:
import io

RESULTS_LOG = ARTIFACTS / "experiments.csv"
LOG_COLUMNS = ["run_name", "owner", "point", "description", "cv_f1", "cv_p", "cv_r", "es_f1", "lb_public", "notes"]


def read_log() -> pd.DataFrame:
    return pd.read_csv(RESULTS_LOG) if RESULTS_LOG.exists() else pd.DataFrame(columns=LOG_COLUMNS)


def log_result(cfg: dict, report: dict, lb_public: float | None = None, notes: str = "") -> pd.DataFrame:
    log = read_log()
    old = log[log.run_name == cfg["run_name"]]
    if lb_public is None and len(old):
        lb_public = old.lb_public.iloc[0]
    row = {"run_name": cfg["run_name"], "owner": cfg["owner"], "point": cfg["point"],
           "description": cfg.get("description", ""), "cv_f1": report["comp_f1"], "cv_p": report["precision"],
           "cv_r": report["recall"], "es_f1": report["es_f1"], "lb_public": lb_public, "notes": notes}
    others = log[log.run_name != cfg["run_name"]]
    log = pd.DataFrame([*others.to_dict("records"), row], columns=LOG_COLUMNS)  # no concat: no all-NA dtype warning
    if not QUICK_RUN:
        log.to_csv(RESULTS_LOG, index=False)
    return log


def kaggle_public_score(message: str, timeout_s: int = 300) -> float | None:
    """Poll our Kaggle submissions until the one with this message has a public score."""
    for _ in range(timeout_s // 15):
        time.sleep(15)
        lines = subprocess.run(["kaggle", "competitions", "submissions", "-c", COMPETITION, "--csv"],
                               capture_output=True, text=True, check=True).stdout.splitlines()
        header = next(i for i, line in enumerate(lines) if "publicScore" in line)  # skip CLI warnings
        subs = pd.read_csv(io.StringIO("\n".join(lines[header:])))
        mine = subs[subs.description.astype(str) == message]
        if len(mine) and pd.notna(mine.publicScore.iloc[0]):  # newest first
            return float(mine.publicScore.iloc[0])
        if len(mine) and "error" in str(mine.status.iloc[0]).lower():
            raise RuntimeError(f"Kaggle rejected the submission: {mine.status.iloc[0]}")
    print("no public score yet; check Kaggle and call log_result(cfg, report, lb_public=...) by hand")
    return None


def submit(cfg: dict, report: dict, notes: str = "") -> float | None:
    path = ARTIFACTS / cfg["run_name"] / "submission.csv"
    message = f"{cfg['run_name']} | CV {report['comp_f1']:.3f}"
    subprocess.run(["kaggle", "competitions", "submit", "-c", COMPETITION, "-f", str(path), "-m", message], check=True)
    lb = kaggle_public_score(message)
    log_result(cfg, report, lb_public=lb, notes=notes)
    print(f"{cfg['run_name']}: public LB = {lb}")
    return lb

### 0.7 BPE 16k vocabulary (copied from `05_tokenization.ipynb`, point 5, owner Ivan)
Point 5's conclusion: *"Use BPE 16k with last-piece labels as the default tokenizer for architecture and optimizer experiments (points 3 and 4)."* This notebook follows that recommendation, so `BpeVocab` is copied here unchanged rather than re-deriving it.

In [ ]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers


class BpeVocab:
    def __init__(self, word_lists, vocab_size: int, min_freq: int = 2):
        tok = Tokenizer(models.BPE(unk_token=UNK_TOKEN, continuing_subword_prefix="##"))
        tok.normalizer = normalizers.Lowercase()
        tok.pre_tokenizer = pre_tokenizers.WhitespaceSplit()  # our words contain no spaces
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=min_freq, show_progress=False,
                                      special_tokens=[PAD_TOKEN, UNK_TOKEN], continuing_subword_prefix="##")
        tok.train_from_iterator((" ".join(words) for words in word_lists), trainer)
        assert tok.token_to_id(PAD_TOKEN) == PAD_ID and tok.token_to_id(UNK_TOKEN) == UNK_ID
        self.tokenizer = tok

    def __len__(self) -> int:
        return self.tokenizer.get_vocab_size()

    def pieces(self, word: str) -> list[str]:
        return self.tokenizer.encode([word], is_pretokenized=True, add_special_tokens=False).tokens

    def encode_batch(self, word_lists, chunk_size: int = 10_000):
        for start in range(0, len(word_lists), chunk_size):
            chunk = word_lists[start:start + chunk_size]
            encodings = self.tokenizer.encode_batch([w or ["."] for w in chunk], is_pretokenized=True,
                                                    add_special_tokens=False)
            for words, enc in zip(chunk, encodings):
                if not words:
                    yield [], [], []
                    continue
                first, last = [None] * len(words), [None] * len(words)
                for pos, w in enumerate(enc.word_ids):
                    if first[w] is None:
                        first[w] = pos
                    last[w] = pos
                assert None not in first, "a word produced no pieces"
                yield enc.ids, first, last

### 0.8 Pluggable optimizer and scheduler (notebook-specific)
`TaggerModule.configure_optimizers` (section 0.4) is hard-coded to AdamW at a fixed lr/weight-decay — fine for points 2, 3 and 5, which never varied it, but not for this notebook. `OptimTaggerModule` subclasses it and reads the optimizer and scheduler from `cfg` instead; `run_experiment_v2` is `run_experiment` (section 0.5) with that one line changed, plus `cfg["steps_per_epoch"]` filled in (needed by schedulers like OneCycle that step per-batch rather than per-epoch).

In [ ]:
import math


def build_optimizer(params, cfg: dict) -> torch.optim.Optimizer:
    name, kw = cfg["optimizer"]["name"], cfg["optimizer"].get("kwargs", {})
    if name == "sgd":
        return torch.optim.SGD(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"], **kw)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"], **kw)
    if name == "adam":
        return torch.optim.Adam(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"], **kw)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"], **kw)
    raise ValueError(f"unknown optimizer: {name}")


def build_scheduler(optimizer: torch.optim.Optimizer, cfg: dict, steps_per_epoch: int):
    """None, or a Lightning `lr_scheduler` dict. `interval` controls whether `.step()` is called once per
    epoch or once per training step; `monitor` is only needed for ReduceLROnPlateau."""
    spec = cfg.get("scheduler")
    if not spec or spec.get("name") is None:
        return None
    name, kw = spec["name"], spec.get("kwargs", {})
    if name == "step":
        sched = torch.optim.lr_scheduler.StepLR(optimizer, **kw)
        return {"scheduler": sched, "interval": "epoch"}
    if name == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["max_epochs"], **kw)
        return {"scheduler": sched, "interval": "epoch"}
    if name == "onecycle":
        sched = torch.optim.lr_scheduler.OneCycleLR(optimizer, total_steps=cfg["max_epochs"] * steps_per_epoch, **kw)
        return {"scheduler": sched, "interval": "step"}
    if name == "plateau":
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", **kw)
        return {"scheduler": sched, "interval": "epoch", "monitor": "es_f1"}
    if name == "linear_warmup":
        total_steps = cfg["max_epochs"] * steps_per_epoch
        warmup_steps = max(1, int(kw.get("warmup_frac", 0.1) * total_steps))

        def lr_lambda(step):
            if step < warmup_steps:
                return step / warmup_steps
            return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        return {"scheduler": sched, "interval": "step"}
    raise ValueError(f"unknown scheduler: {name}")


class OptimTaggerModule(TaggerModule):
    """`TaggerModule` (section 0.4), but the optimizer and scheduler come from `cfg` instead of being
    fixed to AdamW. `cfg["steps_per_epoch"]` is filled in by `run_experiment_v2`."""

    def __init__(self, model: nn.Module, cfg: dict, val_corpora: dict[str, Corpus]):
        super().__init__(model, cfg["lr"], cfg["weight_decay"], val_corpora)
        self.cfg = cfg

    def configure_optimizers(self):
        optimizer = build_optimizer(self.model.parameters(), self.cfg)
        scheduler = build_scheduler(optimizer, self.cfg, self.cfg["steps_per_epoch"])
        return optimizer if scheduler is None else {"optimizer": optimizer, "lr_scheduler": scheduler}

In [ ]:
def run_experiment_v2(cfg: dict, vocab, corpora: dict[str, Corpus], test_ids, force: bool = False) -> dict:
    """`run_experiment` (section 0.5), with `OptimTaggerModule` instead of `TaggerModule` so cfg["optimizer"]
    and cfg["scheduler"] take effect."""
    out = ARTIFACTS / cfg["run_name"]
    if (out / "report.json").exists() and not force:
        return json.loads((out / "report.json").read_text())
    out.mkdir(parents=True, exist_ok=True)
    L.seed_everything(cfg["seed"], workers=True)
    start = time.time()

    encode = lambda name, max_len=None: EncodedDataset(corpora[name], vocab, cfg["alignment"], max_len)
    train_ds, es_ds, cv_ds, test_ds = encode("train", cfg["max_len"]), encode("es"), encode("cv"), encode("test")
    cfg["steps_per_epoch"] = math.ceil(len(train_ds) / cfg["batch_size"])
    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    workers = 2 if DEVICE == "cuda" else 0
    loader = lambda ds, shuffle=False: DataLoader(ds, batch_size=cfg["batch_size"], shuffle=shuffle,
                                                  collate_fn=collate, num_workers=workers)

    model = RNNTagger(len(vocab), **cfg["model"])
    module = OptimTaggerModule(model, cfg, {"es": corpora["es"], "cv": corpora["cv"]})
    loggers = [CSVLogger(ARTIFACTS, name=cfg["run_name"], version="logs")]
    if os.environ.get("WANDB_API_KEY") and not QUICK_RUN:
        loggers.append(WandbLogger(project=WANDB_PROJECT, name=cfg["run_name"], group=NOTEBOOK, config=cfg,
                                   save_dir=str(out)))
    checkpoint = ModelCheckpoint(dirpath=out, filename="best", monitor="es_f1", mode="max", save_top_k=1)
    trainer = L.Trainer(max_epochs=cfg["max_epochs"], accelerator="gpu" if DEVICE == "cuda" else "cpu", devices=1,
                        gradient_clip_val=1.0, log_every_n_steps=10 if QUICK_RUN else 100, num_sanity_val_steps=0,
                        logger=loggers,
                        callbacks=[checkpoint, EarlyStopping(monitor="es_f1", mode="max", patience=cfg["patience"])],
                        enable_progress_bar=False, enable_model_summary=False)
    trainer.fit(module, loader(train_ds, shuffle=True), [loader(es_ds), loader(cv_ds)])

    best = torch.load(checkpoint.best_model_path, map_location="cpu", weights_only=False)
    model.load_state_dict({k.removeprefix("model."): v for k, v in best["state_dict"].items()})
    cv_probs, test_probs = predict(model, cv_ds), predict(model, test_ds)
    save_probs(cv_probs, out / "cv_probs.pkl")
    save_probs(test_probs, out / "test_probs.pkl")
    write_submission(test_ids, decode(corpora["test"], test_probs), out / "submission.csv")

    report = {"run_name": cfg["run_name"], **metric_report(corpora["cv"].gold, decode(corpora["cv"], cv_probs)),
              "es_f1": float(checkpoint.best_model_score), "best_epoch": int(best["epoch"]),
              "epochs_run": trainer.current_epoch, "vocab_size": len(vocab),
              "params": sum(p.numel() for p in model.parameters()), "minutes": (time.time() - start) / 60}
    (out / "report.json").write_text(json.dumps(report, indent=2))
    if len(loggers) > 1:
        loggers[1].experiment.summary.update({f"final_cv_{k}": v for k, v in report.items() if k != "run_name"})
        loggers[1].experiment.finish()
    return report

## 1. Data and base architecture
Same splits and BPE 16k vocabulary as notebook 03. The model architecture is read from point 3's results (best CV F1); only the training recipe (optimizer, scheduler, hyperparameters) varies from here on.

In [ ]:
corpora, test = load_corpora()
start = time.time()
bpe_vocab = BpeVocab(corpora["train"].words, vocab_size=16_000)
print(f"bpe16k vocab: {len(bpe_vocab):,} pieces, built in {time.time() - start:.0f}s")

In [ ]:
ARCH_RESULTS = ARTIFACTS / "results_03_architecture.csv"
if ARCH_RESULTS.exists():
    arch_df = pd.read_csv(ARCH_RESULTS, index_col=0)
    best = arch_df.sort_values("comp_f1", ascending=False).iloc[0]
    BASE_ARCH = {"rnn_type": best.rnn_type, "num_layers": int(best.num_layers),
                "bidirectional": bool(best.direction == "bi")}
    print(f"using point 3's best architecture: {best.name} (cv_f1={best.comp_f1:.3f})")
else:
    BASE_ARCH = {"rnn_type": "gru", "num_layers": 1, "bidirectional": True}
    print("results_03_architecture.csv not found yet — falling back to point 5's recommended "
          "architecture (1-layer BiGRU). Re-run this notebook once point 3 has finished for the real numbers.")
BASE_ARCH

## 2. Optimizers
Fixed architecture (`BASE_ARCH` above), tokenizer (BPE 16k, last piece), and everything else at the point-2/5 baseline (lr 2e-3, weight decay 0.01, batch 256, 6 epochs, patience 2). Two SGD runs: one at the **same lr as AdamW**, to see the gap an adaptive optimizer buys at a fixed learning rate; and one at a **lr actually scaled for SGD** (momentum-SGD typically wants a learning rate 1-2 orders of magnitude larger than Adam-family optimizers), to avoid the unfair comparison of judging SGD at a learning rate that was never tuned for it.

In [ ]:
BASE = {"seed": 42, "max_len": 256, "batch_size": 256, "lr": 2e-3, "weight_decay": 0.01,
        "max_epochs": 1 if QUICK_RUN else 6, "patience": 2, "tokenizer": "bpe16k", "alignment": "last",
        "model": {**BASE_ARCH, "embed_dim": 128, "hidden_dim": 256, "dropout": 0.3}}
SGD_TUNED_LR = 0.1

OPTIMIZER_GRID = [
    ("adamw", {"name": "adamw"}, None),
    ("adam", {"name": "adam"}, None),
    ("rmsprop", {"name": "rmsprop"}, None),
    ("sgd-samelr", {"name": "sgd", "kwargs": {"momentum": 0.9, "nesterov": True}}, None),
    ("sgd-tunedlr", {"name": "sgd", "kwargs": {"momentum": 0.9, "nesterov": True}}, SGD_TUNED_LR),
]
if QUICK_RUN:
    OPTIMIZER_GRID = OPTIMIZER_GRID[:2]


def make_optimizer_config(key: str, opt_spec: dict, lr: float | None) -> dict:
    cfg = {**BASE, "owner": "mykola", "point": 4, "run_name": f"mykola-opt-{key}-bpe16k-last",
           "description": f"{opt_spec['name']} optimizer ({'tuned' if lr else 'baseline'} lr), "
                          f"fixed architecture + BPE16k/last"}
    cfg["optimizer"] = opt_spec
    cfg["scheduler"] = None
    if lr is not None:
        cfg["lr"] = lr
    return cfg

In [ ]:
optimizer_reports = []
for key, opt_spec, lr in OPTIMIZER_GRID:
    cfg = make_optimizer_config(key, opt_spec, lr)
    report = run_experiment_v2(cfg, bpe_vocab, corpora, test.text_id)
    optimizer_reports.append({"optimizer": key, "lr": cfg["lr"], **report})
    print(f"{cfg['run_name']:<28} cv_f1={report['comp_f1']:.3f}  es_f1={report['es_f1']:.3f}  "
          f"({report['minutes']:.1f} min)")

In [ ]:
optimizer_results = pd.DataFrame(optimizer_reports).set_index("optimizer")
optimizer_results.to_csv(ARTIFACTS / "results_04_optimizers.csv")
cols = ["lr", "comp_f1", "precision", "recall", "partial_f1", "es_f1", "best_epoch", "minutes"]
optimizer_results[cols].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(optimizer_results.index, optimizer_results.comp_f1, color="tab:blue", alpha=0.8)
ax.axhline(optimizer_results.loc["adamw", "comp_f1"], color="grey", ls="--", lw=1, label="adamw (baseline)")
ax.set(ylabel="CV F1 (comp_metric)", title="optimizer comparison, fixed architecture + lr (except tuned SGD)")
ax.legend(frameon=False)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 3. Schedulers
Fixed optimizer (AdamW, the team's baseline) so the comparison isolates the scheduler, not a combination of optimizer-and-scheduler effects. `none` reuses section 2's `adamw` run instead of retraining it.

In [ ]:
SCHEDULER_GRID = [
    ("none", None),
    ("step", {"name": "step", "kwargs": {"step_size": 2, "gamma": 0.5}}),
    ("cosine", {"name": "cosine", "kwargs": {}}),
    ("onecycle", {"name": "onecycle", "kwargs": {"max_lr": 5e-3}}),
    ("plateau", {"name": "plateau", "kwargs": {"factor": 0.5, "patience": 1}}),
    ("linear_warmup", {"name": "linear_warmup", "kwargs": {"warmup_frac": 0.1}}),
]
if QUICK_RUN:
    SCHEDULER_GRID = SCHEDULER_GRID[:2]


def make_scheduler_config(key: str, sched_spec: dict | None) -> dict:
    cfg = {**BASE, "owner": "mykola", "point": 4, "run_name": f"mykola-sched-{key}-bpe16k-last",
           "description": f"AdamW + {key} scheduler, fixed architecture + BPE16k/last"}
    cfg["optimizer"] = {"name": "adamw"}
    cfg["scheduler"] = sched_spec
    return cfg

In [ ]:
baseline_report = next(r for r in optimizer_reports if r["optimizer"] == "adamw")

scheduler_reports = []
for key, sched_spec in SCHEDULER_GRID:
    if key == "none":
        report = {k: v for k, v in baseline_report.items() if k != "optimizer"}
    else:
        cfg = make_scheduler_config(key, sched_spec)
        report = run_experiment_v2(cfg, bpe_vocab, corpora, test.text_id)
    scheduler_reports.append({"scheduler": key, **report})
    print(f"{key:<16} cv_f1={report['comp_f1']:.3f}  es_f1={report['es_f1']:.3f}  "
          f"({report.get('minutes', 0):.1f} min)")

In [ ]:
scheduler_results = pd.DataFrame(scheduler_reports).set_index("scheduler")
scheduler_results.to_csv(ARTIFACTS / "results_04_schedulers.csv")
scheduler_results[["comp_f1", "precision", "recall", "partial_f1", "es_f1", "best_epoch"]].round(4)

In [ ]:
BEST_SCHED_KEY = scheduler_results.comp_f1.idxmax()
BEST_SCHED_SPEC = dict(SCHEDULER_GRID)[BEST_SCHED_KEY]
print(f"best scheduler so far: {BEST_SCHED_KEY} (cv_f1={scheduler_results.loc[BEST_SCHED_KEY, 'comp_f1']:.3f})")

## 4. Hyperparameters
One-factor-at-a-time sweeps, holding optimizer = AdamW and scheduler = section 3's best fixed, so each sweep isolates a single hyperparameter against the same baseline rather than compounding changes. The baseline point (lr 2e-3, weight decay 0.01, batch 256, dropout 0.3, hidden 256) is exactly `BASE_SCHED_REPORT` below and is not retrained.

In [ ]:
BASE_SCHED_REPORT = next(r for r in scheduler_reports if r["scheduler"] == BEST_SCHED_KEY)


def make_hparam_config(key: str, **overrides) -> dict:
    cfg = {**BASE, "owner": "mykola", "point": 4, "optimizer": {"name": "adamw"}, "scheduler": BEST_SCHED_SPEC,
           "run_name": f"mykola-hp-{key}-bpe16k-last", "description": f"hyperparameter sweep: {key}"}
    model_overrides = overrides.pop("model", {})
    cfg.update(overrides)
    cfg["model"] = {**BASE["model"], **model_overrides}
    return cfg


HPARAM_GRID = [
    ("lr-5e-4", dict(lr=5e-4)), ("lr-1e-3", dict(lr=1e-3)), ("lr-5e-3", dict(lr=5e-3)), ("lr-1e-2", dict(lr=1e-2)),
    ("wd-0", dict(weight_decay=0.0)), ("wd-0.1", dict(weight_decay=0.1)),
    ("batch-64", dict(batch_size=64)), ("batch-128", dict(batch_size=128)), ("batch-512", dict(batch_size=512)),
    ("dropout-0.1", dict(model=dict(dropout=0.1))), ("dropout-0.5", dict(model=dict(dropout=0.5))),
    ("hidden-128", dict(model=dict(hidden_dim=128))), ("hidden-512", dict(model=dict(hidden_dim=512))),
]
if QUICK_RUN:
    HPARAM_GRID = HPARAM_GRID[:2]

hparam_reports = [{"hyperparameter": "baseline (lr 2e-3, wd 0.01, batch 256, dropout 0.3, hidden 256)",
                   **{k: v for k, v in BASE_SCHED_REPORT.items() if k != "scheduler"}}]
for key, overrides in HPARAM_GRID:
    cfg = make_hparam_config(key, **overrides)
    report = run_experiment_v2(cfg, bpe_vocab, corpora, test.text_id)
    hparam_reports.append({"hyperparameter": key, **report})
    print(f"{key:<14} cv_f1={report['comp_f1']:.3f}  es_f1={report['es_f1']:.3f}  "
          f"({report['minutes']:.1f} min)")

In [ ]:
hparam_results = pd.DataFrame(hparam_reports).set_index("hyperparameter")
hparam_results.to_csv(ARTIFACTS / "results_04_hyperparameters.csv")
hparam_results[["comp_f1", "precision", "recall", "es_f1", "best_epoch"]].round(4)

Hyperparameter curves: CV F1 against each knob, baseline value marked with a dashed line. `hp_score` returns NaN (a gap in the line) for a key that was never run — e.g. under `QUICK_RUN`, where `HPARAM_GRID` is trimmed, so this cell stays usable even on a partial sweep.

In [ ]:
def hp_score(key: str) -> float:
    return hparam_results.loc[key, "comp_f1"] if key in hparam_results.index else np.nan


fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
base_f1 = hparam_results.iloc[0].comp_f1

lr_vals = [5e-4, 1e-3, 2e-3, 5e-3, 1e-2]
lr_scores = [hp_score("lr-5e-4"), hp_score("lr-1e-3"), base_f1, hp_score("lr-5e-3"), hp_score("lr-1e-2")]
axes[0].plot(lr_vals, lr_scores, marker="o")
axes[0].set(xscale="log", xlabel="learning rate", ylabel="CV F1", title="learning rate")

wd_vals = [0.0, 0.01, 0.1]
wd_scores = [hp_score("wd-0"), base_f1, hp_score("wd-0.1")]
axes[1].plot(wd_vals, wd_scores, marker="o")
axes[1].set(xlabel="weight decay", title="weight decay")

batch_vals = [64, 128, 256, 512]
batch_scores = [hp_score("batch-64"), hp_score("batch-128"), base_f1, hp_score("batch-512")]
axes[2].plot(batch_vals, batch_scores, marker="o")
axes[2].set(xscale="log", xlabel="batch size", title="batch size")

drop_vals = [0.1, 0.3, 0.5]
drop_scores = [hp_score("dropout-0.1"), base_f1, hp_score("dropout-0.5")]
axes[3].plot(drop_vals, drop_scores, marker="o")
axes[3].set(xlabel="dropout", title="dropout")

for ax in axes:
    ax.axhline(base_f1, color="grey", ls="--", lw=1)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
hidden_vals = [128, 256, 512]
hidden_scores = [hp_score("hidden-128"), base_f1, hp_score("hidden-512")]
ax.plot(hidden_vals, hidden_scores, marker="o")
ax.axhline(base_f1, color="grey", ls="--", lw=1)
ax.set(xlabel="hidden size", ylabel="CV F1", title="hidden size")
plt.tight_layout()
plt.show()

## 5. Curves and gradient flow across experiments
Two things the assignment explicitly asks for: training curves across the sweep, and a gradient-flow check — here specifically on **how the optimizer/scheduler choice changes the gradient trajectory**, not just the final score.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for key, _, _ in OPTIMIZER_GRID:
    run = f"mykola-opt-{key}-bpe16k-last"
    log = pd.read_csv(ARTIFACTS / run / "logs" / "metrics.csv")
    epochs = log.dropna(subset=["es_f1"]).groupby("epoch")[["es_f1", "cv_f1"]].last()
    axes[0].plot(epochs.index + 1, epochs.es_f1, marker="o", label=key)
    axes[1].plot(epochs.index + 1, epochs.cv_f1, marker="o", label=key)
axes[0].set(title="es F1 by optimizer", xlabel="epoch")
axes[1].set(title="CV F1 by optimizer", xlabel="epoch")
axes[1].legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Gradient-norm trajectory (total parameter-gradient L2 norm, pre-clip — same quantity point 2's baseline plotted) for a few runs chosen to make specific points: **no scheduler vs linear warmup**, to see whether warmup visibly tames the early-training gradient spike that a cold AdamW start can produce; and **the lowest vs highest learning rate in the sweep**, to see instability (large or spiky gradient norms, possibly hugging the clip threshold of 1.0) directly in the gradient signal rather than only inferring it from a worse CV score.

In [ ]:
def total_grad_norm(run: str) -> pd.DataFrame:
    metrics = pd.read_csv(ARTIFACTS / run / "logs" / "metrics.csv")
    grad_cols = [c for c in metrics.columns if c.startswith("grad_2.0_norm/")]
    norms = metrics[["step"] + grad_cols].dropna(how="all", subset=grad_cols)
    total = np.sqrt((norms[grad_cols] ** 2).sum(axis=1))
    return pd.DataFrame({"step": norms.step, "total_grad_norm": total})


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run, label in [("mykola-sched-none-bpe16k-last", "no scheduler"),
                   ("mykola-opt-adamw-bpe16k-last", "no scheduler")]:
    if (ARTIFACTS / run / "logs" / "metrics.csv").exists():
        g = total_grad_norm(run)
        axes[0].plot(g.step, g.total_grad_norm, lw=1, alpha=0.8, label=label)
        break
if (ARTIFACTS / "mykola-sched-linear_warmup-bpe16k-last" / "logs" / "metrics.csv").exists():
    g = total_grad_norm("mykola-sched-linear_warmup-bpe16k-last")
    axes[0].plot(g.step, g.total_grad_norm, lw=1, alpha=0.8, label="linear warmup")
axes[0].axhline(1.0, color="grey", ls="--", lw=1, label="clip threshold")
axes[0].set(title="scheduler: warmup vs none", xlabel="step", ylabel="total ‖∇‖₂ (pre-clip)", yscale="log")
axes[0].legend(frameon=False, fontsize=8)

for key in ("lr-5e-4", "lr-1e-2"):
    run = f"mykola-hp-{key}-bpe16k-last"
    if (ARTIFACTS / run / "logs" / "metrics.csv").exists():
        g = total_grad_norm(run)
        axes[1].plot(g.step, g.total_grad_norm, lw=1, alpha=0.8, label=key)
axes[1].axhline(1.0, color="grey", ls="--", lw=1, label="clip threshold")
axes[1].set(title="learning rate: lowest vs highest swept", xlabel="step", yscale="log")
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**→ what to look for (fill in after running):** does warmup visibly lower the gradient norm in the first few hundred steps compared to no scheduler? Does the highest swept learning rate push the gradient norm noticeably closer to (or above) the clip threshold of 1.0 than the lowest one, i.e. is clipping doing real work there? Tie this back to section 4's CV scores — if a high lr both destabilizes gradients *and* scores worse, that's a consistent story; if it destabilizes gradients but scores fine, clipping is likely doing its job.

_Replace this paragraph with the actual pattern observed once the sweep has run._

## 6. Error analysis
The best overall run from this notebook (optimizer, scheduler and hyperparameters all considered), compared with point 3's best architecture run under the original baseline recipe.

In [ ]:
# Every report dict carries its own "run_name" (from run_experiment_v2, or copied from the run it reuses),
# so the best run across all three sweeps is just the max over all of them, deduplicated by run_name (the
# "none" scheduler and the hyperparameter baseline both reuse the adamw optimizer run).
all_reports = {r["run_name"]: r for r in [*optimizer_reports, *scheduler_reports, *hparam_reports]}
best_report = max(all_reports.values(), key=lambda r: r["comp_f1"])
best_run_name = best_report["run_name"]
print("best run across sections 2-4:", best_run_name, f"(cv_f1={best_report['comp_f1']:.3f})")

In [ ]:
best_pred = decode(corpora["cv"], load_probs(best_run_name, "cv"))
rows = []
for text, gold, pred in zip(corpora["cv"].texts, corpora["cv"].gold, best_pred):
    g, p = {hard_processing(x) for x in gold}, {hard_processing(x) for x in pred}
    if g != p:
        rows.append({"text": text[:100].replace("\n", " ⏎ "), "missed": sorted(g - p), "extra": sorted(p - g),
                     "correct": sorted(g & p)})
print(f"{len(rows)} of {len(best_pred)} CV texts have an error")
pd.DataFrame(rows)

## 7. Conclusions
_Fill in after running the full notebook on Colab — do not report numbers that have not actually been produced by the runs above._

- **Optimizers:** which beat AdamW, if any? Did SGD at the same lr as AdamW badly underperform (expected — AdamW's per-parameter adaptive scaling is usually not matched by vanilla SGD at the same raw lr), and did tuning SGD's lr close most of the gap?
- **Schedulers:** which scheduler helped most over no scheduler, and does that line up with the gradient-flow observation in section 5 (e.g. warmup avoiding an early spike)?
- **Hyperparameters:** summarize each curve from section 4 — is there a clear optimum, a plateau, or is the CV noise (±0.16 at 95%, per point 2) too large to tell? Which knob mattered most?
- **Curves:** does the es-vs-CV gap from points 2/3/5 (train-domain score keeps rising past the point where test-like score peaks) show up again here for any of these runs?
- **Gradient flow:** what section 5 actually showed about warmup and learning-rate extremes.
- **Failed runs, if any:** note any run that diverged, NaN'd, or needed significantly more/fewer epochs than the baseline, and why.
- **Recommended final config:** the specific architecture + optimizer + scheduler + hyperparameters to carry into point 6 (ensembling) — this is the run this notebook should actually submit below.

## 8. Results log and submission
All runs from sections 2-4 go into the shared results log. Review section 7 first, then set `SUBMIT_RUN` to the run you want to spend a Kaggle submission on (or leave it `None` to only update the log).

In [ ]:
run_lookup = {}
for key, opt_spec, lr in OPTIMIZER_GRID:
    run_lookup[f"mykola-opt-{key}-bpe16k-last"] = make_optimizer_config(key, opt_spec, lr)
for key, sched_spec in SCHEDULER_GRID:
    if key != "none":
        run_lookup[f"mykola-sched-{key}-bpe16k-last"] = make_scheduler_config(key, sched_spec)
for key, overrides in HPARAM_GRID:
    run_lookup[f"mykola-hp-{key}-bpe16k-last"] = make_hparam_config(key, **overrides)

for report in [*optimizer_reports, *scheduler_reports, *hparam_reports]:
    run_name = report.get("run_name")
    if run_name in run_lookup:
        log_result(run_lookup[run_name], report, notes="optimization")

SUBMIT_RUN = None  # set to best_run_name (section 6) after reviewing section 7
lb_scores = read_log().set_index("run_name").lb_public
if QUICK_RUN:
    print("quick run: not submitting")
elif SUBMIT_RUN and pd.isna(lb_scores.get(SUBMIT_RUN)) and SUBMIT_RUN in run_lookup:
    report = next(r for r in [*optimizer_reports, *scheduler_reports, *hparam_reports]
                 if r.get("run_name") == SUBMIT_RUN)
    submit(run_lookup[SUBMIT_RUN], report, notes="optimization")
read_log().query("point == 4")